# U-Net on the 2LPT field, and how sparse a survey can get

Two experiments on the slice-based U-Net, both building on the 2LPT upgrade:

1. **Does the sharper target help?** Train one net on a Zel'dovich field and one on a 2LPT field under identical settings, and compare per-class F1.
2. **How sparse can the tracers get?** Take the 2LPT-trained setup and push `nbar` (mean tracers per voxel) down, watching which web classes degrade first.

Same 2D-on-slices design as before (laptop-friendly; the 3D version is the next notebook).

## 0. Setup

In [ ]:
# run once
!pip install torch

In [ ]:
import os, time, numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from numpy.fft import fftn, ifftn, fftfreq
torch.manual_seed(0); np.random.seed(0); torch.set_num_threads(os.cpu_count() or 1)

# --- config ---
N         = 64       # grid per side
L         = 250.0    # box, Mpc/h
sigma_lin = 2.0      # 2LPT strength / nonlinearity
f         = 16       # U-Net width
epochs    = 10
NBAR_BASE = 2.0      # tracers/voxel for the ZA-vs-2LPT comparison
NBAR_SWEEP = [2.0, 1.0, 0.5, 0.25]
CLASS_NAMES  = ['void','sheet','filament','knot']
CLASS_COLORS = ['#08306b','#6baed6','#fd8d3c','#cb181d']
print("torch", torch.__version__, "| threads", torch.get_num_threads())

## 1. Field generator, tracers, labels, and the U-Net

`lpt_delta(order=1)` is Zel'dovich, `order=2` is 2LPT. `make_cube` draws Poisson tracers with rate proportional to (1+δ) — the sparse view the net trains on.

In [ ]:
def Pk_bbks(k, ns=0.96, G=0.2):
    k=np.asarray(k,float); q=np.where(k>0,k/G,1e-10)
    T=(np.log(1+2.34*q)/(2.34*q))*(1+3.89*q+(16.1*q)**2+(5.46*q)**3+(6.71*q)**4)**(-0.25)
    return k**ns*T**2
def cic(pos,N,L):
    g=np.zeros((N,N,N)); s=(pos%L)/(L/N); i=np.floor(s).astype(int)%N; fr=s-np.floor(s)
    for dx in (0,1):
        for dy in (0,1):
            for dz in (0,1):
                wx=fr[:,0] if dx else 1-fr[:,0]; wy=fr[:,1] if dy else 1-fr[:,1]; wz=fr[:,2] if dz else 1-fr[:,2]
                np.add.at(g,((i[:,0]+dx)%N,(i[:,1]+dy)%N,(i[:,2]+dz)%N),wx*wy*wz)
    return g
def lpt_delta(N,L,sigma_lin,seed,order):
    kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0
    rng=np.random.default_rng(seed)
    dk=fftn(rng.standard_normal((N,N,N)))*np.sqrt(Pk_bbks(np.sqrt(k2))); dk[0,0,0]=0
    d=ifftn(dk).real; d*=sigma_lin/d.std(); dk=fftn(d); src=dk.copy()
    if order==2:
        pij=lambda a,b: ifftn(Kv[a]*Kv[b]/k2*dk).real
        pxx,pyy,pzz=pij(0,0),pij(1,1),pij(2,2); pxy,pxz,pyz=pij(0,1),pij(0,2),pij(1,2)
        S2=pxx*pyy+pxx*pzz+pyy*pzz-pxy**2-pxz**2-pyz**2
        src=dk+(3.0/7.0)*fftn(S2)
    disp=np.array([ifftn(1j*Kv[i]*src/k2).real for i in range(3)])
    q=np.arange(N)*(L/N); QX,QY,QZ=np.meshgrid(q,q,q,indexing='ij'); Q=[QX,QY,QZ]
    pos=np.stack([(Q[i]+disp[i]).ravel() for i in range(3)],axis=1)
    rho=cic(pos,N,L); return rho/rho.mean()-1.0
def smooth(d,L,R):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    return ifftn(fftn(d)*np.exp(-0.5*(KX**2+KY**2+KZ**2)*R**2)).real
def tweb(d,L,lam=0.3):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0; dk=fftn(d); T=np.zeros((N,N,N,3,3))
    for i in range(3):
        for j in range(i,3):
            t=ifftn((Kv[i]*Kv[j]/k2)*dk).real; T[...,i,j]=t
            if i!=j: T[...,j,i]=t
    return np.sum(np.linalg.eigvalsh(T)>lam,-1).astype(np.int64)
def make_cube(N,L,seed,nbar,sigma_lin,order):
    d=lpt_delta(N,L,sigma_lin,seed,order); lab=tweb(smooth(d,L,3.0),L,0.3)
    rng=np.random.default_rng(seed+999)
    counts=rng.poisson(nbar*np.clip(1+d,0,None)); dsp=counts/counts.mean()-1.0
    return dsp.astype(np.float32),lab
def to_slices(dsp,lab):
    X,Y=[],[]
    for ax in range(3):
        ds=np.moveaxis(dsp,ax,0); ls=np.moveaxis(lab,ax,0)
        for k in range(ds.shape[0]): X.append(ds[k]); Y.append(ls[k])
    return np.array(X),np.array(Y)

def double_conv(ci,co):
    return nn.Sequential(nn.Conv2d(ci,co,3,padding=1),nn.BatchNorm2d(co),nn.ReLU(True),
                         nn.Conv2d(co,co,3,padding=1),nn.BatchNorm2d(co),nn.ReLU(True))
class UNet2D(nn.Module):
    def __init__(self,f=16,nc=4):
        super().__init__(); self.e1=double_conv(1,f); self.e2=double_conv(f,2*f); self.b=double_conv(2*f,4*f)
        self.u2=nn.ConvTranspose2d(4*f,2*f,2,2); self.d2=double_conv(4*f,2*f)
        self.u1=nn.ConvTranspose2d(2*f,f,2,2); self.d1=double_conv(2*f,f); self.o=nn.Conv2d(f,nc,1); self.p=nn.MaxPool2d(2)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.p(e1)); b=self.b(self.p(e2))
        d2=self.d2(torch.cat([self.u2(b),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1)); return self.o(d1)
def per_class_f1(pred,true,nc=4):
    o=[]
    for c in range(nc):
        tp=((pred==c)&(true==c)).sum(); fp=((pred==c)&(true!=c)).sum(); fn=((pred!=c)&(true==c)).sum()
        o.append(2*tp/(2*tp+fp+fn+1e-9))
    return o
def train_eval(nbar, order, sigma_lin=sigma_lin, N=N, L=L, f=f, epochs=epochs):
    Xtr,Ytr=to_slices(*make_cube(N,L,1,nbar,sigma_lin,order))
    Xva,Yva=to_slices(*make_cube(N,L,2,nbar,sigma_lin,order))
    mu,sd=Xtr.mean(),Xtr.std()
    Xt=torch.tensor(((Xtr-mu)/sd)[:,None]); Yt=torch.tensor(Ytr)
    Xv=torch.tensor(((Xva-mu)/sd)[:,None]); Yv=torch.tensor(Yva)
    fr=np.bincount(Ytr.ravel(),minlength=4); w=(1.0/(fr+1)); w=(w/w.sum()*4).astype(np.float32)
    net=UNet2D(f); opt=torch.optim.Adam(net.parameters(),1e-3); crit=nn.CrossEntropyLoss(weight=torch.tensor(w))
    bs,n=16,len(Xt)
    for ep in range(epochs):
        net.train(); perm=torch.randperm(n)
        for i in range(0,n,bs):
            idx=perm[i:i+bs]; opt.zero_grad(); crit(net(Xt[idx]),Yt[idx]).backward(); opt.step()
    net.eval()
    with torch.no_grad(): pv=net(Xv).argmax(1)
    return (pv==Yv).float().mean().item(), per_class_f1(pv.numpy(),Yv.numpy())
print("ready")

## 2. Does the 2LPT target help? (nbar = 2.0)

Two nets, identical except the field they learn from.

In [ ]:
t0=time.time()
acc_za, f1_za = train_eval(NBAR_BASE, order=1)
acc_2l, f1_2l = train_eval(NBAR_BASE, order=2)
print(f"Zel'dovich : valAcc {acc_za:.3f}   F1  void {f1_za[0]:.2f}  sheet {f1_za[1]:.2f}  filament {f1_za[2]:.2f}  knot {f1_za[3]:.2f}")
print(f"2LPT       : valAcc {acc_2l:.3f}   F1  void {f1_2l[0]:.2f}  sheet {f1_2l[1]:.2f}  filament {f1_2l[2]:.2f}  knot {f1_2l[3]:.2f}")
print(f"({time.time()-t0:.0f}s)")

## 3. How sparse can the survey get? (2LPT, sweep nbar)

Retrain a fresh net at each tracer density.

In [ ]:
sweep = {NBAR_BASE: (acc_2l, f1_2l)}          # reuse the 2LPT/nbar=2.0 run above
for nb in [x for x in NBAR_SWEEP if x != NBAR_BASE]:
    t0=time.time(); acc,F = train_eval(nb, order=2)
    sweep[nb]=(acc,F); print(f"nbar {nb:4.2f}: valAcc {acc:.3f}  F1 V{F[0]:.2f} S{F[1]:.2f} F{F[2]:.2f} K{F[3]:.2f}  ({time.time()-t0:.0f}s)")
nbars = sorted(sweep, reverse=True)

## 4. Results

Left: the 2LPT field gives equal-or-better F1 on every class. Right: as tracers thin out, knots (and then filaments) fall away first, while voids stay recoverable.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(14,5))
x=np.arange(4); wd=0.38
ax[0].bar(x-wd/2,f1_za,wd,label="Zel'dovich",color='#9ecae1')
ax[0].bar(x+wd/2,f1_2l,wd,label='2LPT',color='#fb6a4a')
ax[0].set_xticks(x); ax[0].set_xticklabels(CLASS_NAMES); ax[0].set_ylabel('F1'); ax[0].set_ylim(0,1)
ax[0].set_title('Per-class F1 at nbar=2.0'); ax[0].legend()
F=np.array([sweep[nb][1] for nb in nbars])
for c in range(4): ax[1].plot(nbars,F[:,c],'o-',color=CLASS_COLORS[c],label=CLASS_NAMES[c])
ax[1].plot(nbars,[sweep[nb][0] for nb in nbars],'k--',label='overall acc')
ax[1].set_xscale('log'); ax[1].set_xlabel('mean tracers per voxel (nbar)'); ax[1].set_ylabel('F1 / accuracy')
ax[1].set_title('Recovery vs survey sparsity'); ax[1].invert_xaxis(); ax[1].legend()
plt.tight_layout(); plt.show()

## What this shows, and the next step

- The 2LPT field is both more realistic *and* slightly easier to recover from — sharper structures imprint more clearly on the sparse tracers.
- There's a sparsity floor: below ~1 tracer per voxel, knots and thin filaments stop being recoverable, which mirrors why real surveys need adequate galaxy density to map the fine web.
- **Next: 3D.** Slices throw away the third dimension. The 3D U-Net (next notebook) learns on sub-cubes and is where a GPU starts to matter.